In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/test.csv")
sample_df = pd.read_csv("../input/house-prices-advanced-regression-techniques/sample_submission.csv")

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
sample_df

In [ ]:
# alldata = ["LotArea","YrSold","MoSold","MiscVal","PoolArea","ScreenPorch","3SsnPorch","EnclosedPorch","OpenPorchSF","WoodDeckSF","LowQualFinSF","HalfBath","LotArea","OverallCond","MSSubClass"]
# X = train_df[alldata]
# y = train_df["SalePrice"]#オリジナル
# # y = np.log1p(train_df["SalePrice"])#対数変換することで評価関数と同じスケールに！！
# X_test = test_df[alldata]

# #ダメ：SaleCondition、SaleType,MiscFeature,Fence,PoolQC,PavedDrive,GarageCond,GarageQual,GarageArea,GarageCars,GarageFinish,GarageYrBlt,GarageType,FireplaceQu,Functional,
# #KitchenQual,Kitchen,Bedroom,BsmtHalfBath,BsmtFullBath,Electrical,CentralAir,HeatingQC,Heating,TotalBsmtSFBsmtUnfSF,,BsmtFinSF2,BsmtFinType2,BsmtFinSF1,BsmtFinType1,BsmtExposure
# #BsmtCond,BsmtQual,Foundation,ExterCond,ExterQual,MasVnrArea,MasVnrType,Exterior2nd,Exterior1st,RoofMatl,RoofStyle,HouseStyle,BldgType,Condition2,Condition1,Neighborhood,LandSlope
# #LotConfig,Utilities,LandContour,LotShape,Alley,Street,LotFrontage,MSZoning

# #なくて良いかも：MSSubClass"YearRemodAdd","YearBuilt","OverallCond","OverallQual","LotArea"、"TotRmsAbvGrd","HalfBath","FullBath","GrLivArea","LowQualFinSF","2ndFlrSF","1stFlrSF"、,"Fireplaces"

In [ ]:
import numpy as np
import pandas as pd
y = np.log1p(train_df["SalePrice"])

train_ID = train_df['Id']
test_ID = test_df['Id']
ntrain = train_df.shape[0]
# データの合体をして、各学習データごとの欠損値を統一して埋めることとする後でtrainとtestに分割し直すために、それぞれのIDとtrainの行数を記憶しておく

# SalePriceはyとして使ったので、他の特徴量と混ざらないようにtrainから削除
train_df = train_df.drop("SalePrice", axis=1)

# ---- trainとtestをガッチャンコ！ ----
all_data = pd.concat((train_df, test_df)).reset_index(drop=True)
# Idは予測に不要なので削除
all_data = all_data.drop('Id', axis=1)

# ---- 整理整頓その１：数値データの欠損値（空欄）を埋める ----
# まずは数値タイプの列をすべて取得
numeric_features = all_data.select_dtypes(include=np.number).columns
# 各列の「中央値」で空欄を埋める（平均値より外れ値に強いので安全）
for col in numeric_features:
    all_data[col] = all_data[col].fillna(all_data[col].median())

# ---- 整理整頓その２：文字データ（カテゴリ）をコンピュータ語に翻訳 ----
# 'MSZoning'のような文字データを、01に変換
# 例：「RL」なら「MSZoning_RL」列が1に、それ以外が0になる
all_data = pd.get_dummies(all_data)

# ---- 整理整頓が終わったので、元に戻す ----
X = all_data[:ntrain]
X_test = all_data[ntrain:]


In [ ]:
# これで X, y, X_test の準備が完了しました！
print("前処理完了！")
print("X (学習データの特徴量)のサイズ:", X.shape)
print("y (学習データのターゲット)のサイズ:", y.shape)
print("X_test (テストデータの特徴量)のサイズ:", X_test.shape)


In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=0)

In [ ]:
import optuna
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

In [ ]:
# Optunaの目的関数を定義 (最小化したいスコア（RMSLE）を返す)
def objective(trial):
    
    # 探索するハイパーパラメータの範囲を定義
    params = {
        'objective': 'regression_l1', # 回帰。MAE (L1) は外れ値に強い傾向がある
        'metric': 'rmse', # 評価指標。np.log1p(y) のRMSE = RMSLE
        'n_estimators': 10000, # 多めに設定し、Early Stoppingで止める
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 50),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 30),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1, # ログを非表示
    }

    # KFold交差検証 (5分割)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_scores = [] # 各Foldの検証スコアを格納するリスト

    # X, y は cell 167 で作成済みのものを前提とします
    for fold, (train_index, valid_index) in enumerate(kf.split(X, y)):
        X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
        y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]
        
        model = lgb.LGBMRegressor(**params)
        
        model.fit(X_train, y_train,
                  eval_set=[(X_valid, y_valid)],
                  eval_metric='rmse',
                  callbacks=[lgb.early_stopping(100, verbose=False)]) # 100ラウンド改善しなければ停止
        
        valid_preds = model.predict(X_valid)
        # yは対数変換済みなので、RMSEを計算すればそれがRMSLEになる
        fold_rmsle = np.sqrt(mean_squared_error(y_valid, valid_preds))
        oof_scores.append(fold_rmsle)

    # 5-Foldの平均RMSLEをOptunaが最小化すべきスコアとして返す
    return np.mean(oof_scores)

In [ ]:
# Optunaの実行
# n_trialsは試行回数。時間があれば100回以上推奨
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50) # まずは50回試行

print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)
print('Best value (RMSLE):', study.best_value)

# 最適なパラメータを取得
best_params = study.best_params
# n_estimators は Early Stopping で決まるため、Optunaの探索結果から削除（あるいは固定値で上書き）
# best_params['n_estimators'] = 10000 # Early Stopping前提で多めに設定
best_params['objective'] = 'regression_l1'
best_params['metric'] = 'rmse'
best_params['random_state'] = 42
best_params['n_jobs'] = -1
best_params['verbose'] = -1

# n_estimators は Early Stopping で決めるため、10000 (多め) に設定
best_params['n_estimators'] = 10000

### 最適パラメータでの学習とFeature Importanceの確認

In [ ]:
# KFoldで学習し、テストデータを予測する
kf = KFold(n_splits=5, shuffle=True, random_state=42)
test_preds_log = [] # テストデータへの予測値 (対数スケール) を格納
models = [] # 各Foldの学習済みモデルを格納
feature_importances = pd.DataFrame() # 特徴量重要度を格納

print("Starting 5-Fold CV and prediction with best params...")

for fold, (train_index, valid_index) in enumerate(kf.split(X, y)):
    print(f"--- Fold {fold+1}/5 --- ")
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]
    
    model = lgb.LGBMRegressor(**best_params)
    
    model.fit(X_train, y_train,
              eval_set=[(X_valid, y_valid)],
              eval_metric='rmse',
              callbacks=[lgb.early_stopping(100, verbose=False)])
    
    # テストデータを予測 (対数スケール)
    test_pred_log = model.predict(X_test)
    test_preds_log.append(test_pred_log)
    models.append(model)
    
    # Feature Importance を蓄積
    _df = pd.DataFrame()
    _df['feature'] = X.columns
    _df['importance'] = model.feature_importances_
    _df['fold'] = fold
    feature_importances = pd.concat([feature_importances, _df], axis=0)

print("Training complete.")

# 5つのモデルの予測値の平均を取る (アンサンブル)
final_pred_test_log = np.mean(test_preds_log, axis=0)

### Feature Importanceの可視化

In [ ]:
# 特徴量重要度のFold平均を計算
mean_importance = feature_importances.groupby('feature').mean()['importance'].reset_index()
mean_importance = mean_importance.sort_values(by='importance', ascending=False)

import matplotlib.pyplot as plt
import seaborn as sns

# 上位50件を表示
plt.figure(figsize=(10, 15))
sns.barplot(x='importance', y='feature', data=mean_importance.head(50))
plt.title('Feature Importance (Top 50)')
plt.savefig('feature_importance.png') # ファイルに保存
print("Feature importance plot saved to feature_importance.png")
# plt.show() # スクリプト実行環境では表示されない場合があるため、ファイル保存を推奨

# 重要度が低い特徴量を確認 (例: 1以下)
low_importance_features = mean_importance[mean_importance['importance'] <= 1]['feature']
print(f"\nLow importance (<= 1) features ({len(low_importance_features)}):")
print(low_importance_features.tolist())


### モデル構築

In [ ]:
# # from sklearn.linear_model import LinearRegression
# # LightGBM
# import lightgbm as lgb
# #検索した結果、このアルゴリズムはなんか良いらしい！後で調べる事、、、
# #高速で動く決定木の一種で「もっとも効果がありそう！」な部分から集中して強化する機能を持っているらしい、、、
# #特徴量が多くても、重要なものを見つけ出すらしいので今回の問題で最適化はわからない

# # モデルを定義
# model = lgb.LGBMRegressor(objective='regression',
#                           num_leaves=10,#一本の木が持てる葉の最大数（知識の複雑さ）。大きいほど複雑な関係を学べるが、過学習しやすくなる。
#                           learning_rate=0.05,#一つ前の木の間違いから、どれくらいのペースで学ぶか。小さいほど丁寧に学習するが、たくさんの木（n_estimators）が必要になる。
#                           n_estimators=450,#木を何本作るか。増やしすぎると過学習（練習問題の解きすぎ）になる。
#                           max_depth=0,#木の最大の深さ。num_leavesと同様にモデルの複雑さをコントロールする。あんま結果に関わんなかったので0
#                           random_state=42) # random_stateで結果を固定

# #https://qiita.com/R1ck29/items/50ba7fa5afa49e334a8f
# #https://qiita.com/studio_haneya/items/e70e835c26524d506e19

In [ ]:
# model = LinearRegression()

In [ ]:
# model.fit(X_train, y_train)

### モデル適用と評価

In [ ]:
# pred_train = model.predict(X_train)
# pred_valid = model.predict(X_valid)
# pred_test = model.predict(X_test)

In [ ]:
# from sklearn.metrics import mean_squared_error

In [ ]:
# print(mean_squared_error(y_train, pred_train), mean_squared_error(y_valid, pred_valid))

### Submission

In [ ]:
# final_predictions = np.expm1(pred_test)

In [ ]:
# 予測値を元のスケールに戻す (np.log1p の逆変換)
final_predictions = np.expm1(final_pred_test_log)

sub_df = pd.DataFrame({
    "Id": sample_df["Id"], 
    "SalePrice": final_predictions
})

In [ ]:
# sub_df = pd.DataFrame({
#     "Id": sample_df["Id"], 
#     "SalePrice": final_predictions
# })
# #投稿ファイルに合う形式のcsvに整える

In [ ]:
# sub_df.to_csv("simple.csv", index=False)
# 投稿ファイルに合う形式のcsvに整える
submission_filename = "optuna_submission.csv"
sub_df.to_csv(submission_filename, index=False)
print(f"Submission file created: {submission_filename}")
print("Script finished.")

In [ ]:
# from sklearn.metrics import mean_squared_error

# # y_valid(正解)とpred_valid(予測)は、どちらも対数スケール
# # なので、このRMSEを計算するだけで、それがRMSLEの値と同じ意味になる
# rmsle = np.sqrt(mean_squared_error(y_valid, pred_valid))
# print(f"手元での検証スコア (RMSLE): {rmsle}")
# # from sklearn.metrics import mean_squared_log_error
# # rmsle1 = np.sqrt(mean_squared_log_error(y_valid, pred_valid))
# # print("RMSlE１:", rmsle1)
# # # rmsle_results.append(rmsle)
# # # models.append(model)
